# Stage A / NB 01 — MIDRC data inventory, image QC, and label reconciliation

Protocol reference: Section 9 Stage A NB 01; addresses referee 2b (data detail) and
supplies the exclusion logic that Table 1 and the mRALE endpoint definition depend on.

Source of truth: `covid_midrc_dataset.csv` (2,581 rows, one row per frontal PNG).

What this notebook establishes
1. Every `path` resolves on the cluster after `/vf/users` -> `/data` rewriting.
2. Every image opens, and its dimensions, mode, and bit depth are recorded.
3. A 64-bit difference hash per image, so that NB 03 can prove external cohorts do not
   duplicate internal images, and so that near-duplicate frontal views inside MIDRC are
   visible rather than assumed absent.
4. mRALE label integrity: component ranges, qualitative-to-numerical consistency, and
   `extent_right*density_right + extent_left*density_left == mRALE Score`.
5. The grouping key for NB 02's folds, and how many images share one.

Outputs (under `stage_A/nb01_inventory/`)
- `midrc_manifest.csv`          one row per image, everything downstream reads this
- `image_hashes.csv`            filename, dhash64, sha256 of file bytes
- `label_consistency_exceptions.csv`
- `excluded_cases_log.csv`
- `data_quality_report.md`
- `inventory_summary.json`
- `gate_nb01.json`

Gate
- 0 unresolved image paths, 0 unreadable images, 0 out-of-range components.
- The component-vs-annotated-total mismatch count is recorded explicitly and must be 0
  for the primary cohort, or the mismatching rows must be listed in the exceptions file.

## 1. Imports and configuration

Paths come from `stage_a_paths.json` written by NB 00. If that file is absent, NB 00 has not
been run on this node, and the fallback constants below are used with a printed warning.

In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import sys
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageStat

Image.MAX_IMAGE_PIXELS = None  # Radiographs legitimately exceed the decompression-bomb limit.

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_PROJECT_ROOT = Path("/data/liangz2/openi/midrc")
FALLBACK_STAGE_A_DIR = FALLBACK_PROJECT_ROOT / "tetci_resubmit" / "stage_A"
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
]

stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Loaded path contract from:", candidate)
        break

if stage_paths is None:
    print("WARNING: stage_a_paths.json not found. Run NB 00 first.")
    print("Using fallback constants; dataset hash verification is skipped.")
    stage_paths = {
        "project_root": str(FALLBACK_PROJECT_ROOT),
        "stage_a_dir": str(FALLBACK_STAGE_A_DIR),
        "datasets": {},
        "dataset_hashes": {},
        "image_path_rewrites": {"/vf/users/liangz2/openi": "/data/liangz2/openi"},
        "nb_output_dirs": {},
    }

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
NB01_DIR = Path(stage_paths.get("nb_output_dirs", {}).get(
    "nb01_inventory", STAGE_A_DIR / "nb01_inventory"))
NB01_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_PATH_REWRITES = OrderedDict(stage_paths["image_path_rewrites"])

MIDRC_CSV = stage_paths["datasets"].get("covid_midrc_dataset_csv")
if MIDRC_CSV is None:
    for candidate in [STAGE_A_DIR.parent / "datasets" / "covid_midrc_dataset.csv",
                      PROJECT_ROOT / "datasets" / "covid_midrc_dataset.csv",
                      Path.cwd() / "covid_midrc_dataset.csv"]:
        if candidate.is_file():
            MIDRC_CSV = str(candidate)
            break
MIDRC_CSV = Path(MIDRC_CSV) if MIDRC_CSV else None
if MIDRC_CSV is None or not MIDRC_CSV.is_file():
    raise FileNotFoundError(
        "covid_midrc_dataset.csv could not be located. Run NB 00 or set MIDRC_CSV manually."
    )

# QC thresholds. Values are choices, so they are recorded in inventory_summary.json.
MIN_IMAGE_SIDE = 256              # Below this, mRALE extent judgement is not credible.
MAX_ASPECT_RATIO = 2.5            # Frontal CXRs are near-square; 2.5 flags a cropped panel.
NEAR_DUPLICATE_HAMMING = 3        # dhash64 distance at or below this is a near-duplicate.

# How near-duplicate pairs that span two study directories affect the fold-grouping key.
#   "conservative" (default) -- merge the two groups UNLESS the merge would create a group
#                               holding both a PCR-positive and a PCR-negative image. A merge
#                               that manufactures contradictory labels is far more likely to
#                               be a perceptual-hash collision than a true duplicate, and
#                               forcing it through would corrupt the stratification for no
#                               leakage benefit.
#   "aggressive"             -- merge every cross-study near-duplicate pair regardless of
#                               label agreement. Maximum leakage safety, at the cost of
#                               mixed-label groups that NB 02 must resolve by majority vote.
#   "sha256_only"            -- merge only byte-identical files. Strongest evidence, weakest
#                               coverage. Use this if Section 5b shows the perceptual hash is
#                               not discriminative on this cohort.
#   "none"                   -- no merging; group_id is exactly the study directory.
# Section 5b measures the hash's false-positive rate so this is an evidence-based choice
# rather than a guess.
NEAR_DUPLICATE_MERGE_POLICY = "conservative"
BLANK_IMAGE_STDDEV = 1.0          # Effectively uniform pixels.

# Rows carrying these quality_issue values are FLAGGED, not dropped, by default.
# The primary analysis keeps them; E-series sensitivity analyses can exclude them.
QUALITY_FLAGS_TO_EXCLUDE = set()  # e.g. {"Incomplete (Missing Side)"} to exclude.

SEVERITY_BANDS = [(0, 0, "none"), (1, 10, "mild"), (11, 18, "moderate"), (19, 24, "severe")]

EXTENT_TO_NUMERIC = {"": 0, "0%": 0, "<=25%": 1, "25-50%": 2, "51-75%": 3, ">75%": 4}
DENSITY_TO_NUMERIC = {"": 0, "Hazy": 1, "Moderate": 2, "Dense": 3}

print("MIDRC CSV:", MIDRC_CSV)
print("Output dir:", NB01_DIR)

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_hash = stage_paths.get("dataset_hashes", {}).get("covid_midrc_dataset_csv")
actual_hash = sha256_file(MIDRC_CSV)
print("covid_midrc_dataset.csv sha256:", actual_hash)
if expected_hash is None:
    print("No recorded hash to compare against (NB 00 not run, or file added later).")
elif expected_hash != actual_hash:
    raise RuntimeError(
        "covid_midrc_dataset.csv has changed since NB 00 recorded it.\n"
        f"  NB 00: {expected_hash}\n  now:   {actual_hash}\n"
        "Re-run NB 00, then re-run NB 01 and NB 02. Folds must never be built from a "
        "different file version than the one the manifest describes."
    )
else:
    print("Hash matches the NB 00 manifest.")

## 2. Read the CSV and normalise fields

Nothing is dropped here. Every row is carried forward with explicit status columns so that
the exclusion decisions are visible in one file rather than buried in filter expressions.

In [ ]:
def rewrite_image_path(path):
    text = str(path).strip()
    candidates = [Path(text)]
    for old_prefix, new_prefix in IMAGE_PATH_REWRITES.items():
        if text.startswith(old_prefix):
            candidates.append(Path(new_prefix + text[len(old_prefix):]))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate), True
    # Return the best-guess rewritten path so the log shows what was attempted.
    return str(candidates[-1]), False


def to_int(value):
    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none"}:
        return None
    try:
        return int(float(text))
    except ValueError:
        return None


def severity_band(total):
    if total is None:
        return None
    for low, high, name in SEVERITY_BANDS:
        if low <= total <= high:
            return name
    return "out_of_range"


with MIDRC_CSV.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    csv_fieldnames = list(reader.fieldnames or [])
    raw_rows = list(reader)

REQUIRED_COLUMNS = {
    "filename", "path", "covid_positive", "mRALE Score",
    "extent_right", "density_right", "extent_left", "density_left",
    "extent_right_numerical", "density_right_numerical",
    "extent_left_numerical", "density_left_numerical",
    "quality_issue", "sex", "race", "ethnicity", "fold",
}
missing_columns = REQUIRED_COLUMNS - set(csv_fieldnames)
if missing_columns:
    raise ValueError(f"covid_midrc_dataset.csv is missing columns: {sorted(missing_columns)}")

print(f"Rows: {len(raw_rows):,}")
print("Columns:", csv_fieldnames)

In [ ]:
records = []
for line_number, row in enumerate(raw_rows, start=2):  # start=2 accounts for the header
    resolved_path, path_ok = rewrite_image_path(row["path"])
    components = {
        "extent_right_numerical": to_int(row["extent_right_numerical"]),
        "density_right_numerical": to_int(row["density_right_numerical"]),
        "extent_left_numerical": to_int(row["extent_left_numerical"]),
        "density_left_numerical": to_int(row["density_left_numerical"]),
    }
    total_annotated = to_int(row["mRALE Score"])
    right = (components["extent_right_numerical"] * components["density_right_numerical"]
             if None not in (components["extent_right_numerical"],
                             components["density_right_numerical"]) else None)
    left = (components["extent_left_numerical"] * components["density_left_numerical"]
            if None not in (components["extent_left_numerical"],
                            components["density_left_numerical"]) else None)
    total_derived = right + left if None not in (right, left) else None

    records.append({
        "csv_line": line_number,
        "filename": row["filename"].strip(),
        "source_path": row["path"].strip(),
        "image_path": resolved_path,
        "path_resolved": path_ok,
        "study_dir": str(Path(resolved_path).parent),
        "study_uid": Path(resolved_path).parent.name,
        "covid_positive": row["covid_positive"].strip(),
        "sex": row["sex"].strip(),
        "race": row["race"].strip(),
        "ethnicity": row["ethnicity"].strip(),
        "quality_issue": row["quality_issue"].strip(),
        "legacy_fold": to_int(row["fold"]),
        "extent_right_text": row["extent_right"].strip(),
        "density_right_text": row["density_right"].strip(),
        "extent_left_text": row["extent_left"].strip(),
        "density_left_text": row["density_left"].strip(),
        **components,
        "mrale_right": right,
        "mrale_left": left,
        "mrale_total_derived": total_derived,
        "mrale_total_annotated": total_annotated,
        "severity_band": severity_band(total_annotated),
    })

frame = pd.DataFrame.from_records(records)
print(f"Parsed {len(frame):,} records")
print()
print("COVID (PCR):", dict(frame["covid_positive"].value_counts()))
print("Severity band:", dict(frame["severity_band"].value_counts()))
print("Sex:", dict(frame["sex"].value_counts()))
print("Quality issue:", dict(frame["quality_issue"].replace("", "<none>").value_counts()))
print()
prevalence = (frame["covid_positive"] == "Yes").mean()
print(f"PCR-positive prevalence: {prevalence:.4f}")
print("This is the imbalance behind the specificity collapse in the rejected submission "
      "(fold-mean specificity 0.413). Experiment family E8 targets it directly, and AUROC "
      "not accuracy is the primary detection endpoint.")

## 3. Label integrity

Four independent checks. Each produces rows in `label_consistency_exceptions.csv` with a
`check` column, so a single file answers "what is wrong with the labels".

In [ ]:
exceptions = []


def add_exception(record, check, detail):
    exceptions.append({
        "csv_line": record["csv_line"],
        "filename": record["filename"],
        "check": check,
        "detail": detail,
    })


EXTENT_RANGE = (0, 4)
DENSITY_RANGE = (0, 3)

for record in records:
    # Check 1: component presence and range.
    for field, (low, high) in [
        ("extent_right_numerical", EXTENT_RANGE),
        ("extent_left_numerical", EXTENT_RANGE),
        ("density_right_numerical", DENSITY_RANGE),
        ("density_left_numerical", DENSITY_RANGE),
    ]:
        value = record[field]
        if value is None:
            add_exception(record, "component_missing", f"{field} is empty")
        elif not (low <= value <= high):
            add_exception(record, "component_out_of_range", f"{field}={value} not in [{low},{high}]")

    # Check 2: qualitative text agrees with the numerical column.
    for text_field, numeric_field, mapping in [
        ("extent_right_text", "extent_right_numerical", EXTENT_TO_NUMERIC),
        ("extent_left_text", "extent_left_numerical", EXTENT_TO_NUMERIC),
        ("density_right_text", "density_right_numerical", DENSITY_TO_NUMERIC),
        ("density_left_text", "density_left_numerical", DENSITY_TO_NUMERIC),
    ]:
        text = record[text_field]
        expected = mapping.get(text, "UNKNOWN")
        if expected == "UNKNOWN":
            add_exception(record, "qualitative_unmapped",
                          f"{text_field}={text!r} is not in the coding dictionary")
        elif record[numeric_field] is not None and expected != record[numeric_field]:
            add_exception(record, "qualitative_numeric_mismatch",
                          f"{text_field}={text!r} implies {expected} "
                          f"but {numeric_field}={record[numeric_field]}")

    # Check 3: the annotated total equals the product-sum.
    if record["mrale_total_annotated"] is None:
        add_exception(record, "total_missing", "mRALE Score is empty")
    elif record["mrale_total_derived"] is None:
        add_exception(record, "total_not_derivable", "one or more components are missing")
    elif record["mrale_total_annotated"] != record["mrale_total_derived"]:
        add_exception(record, "total_mismatch",
                      f"annotated={record['mrale_total_annotated']} "
                      f"derived={record['mrale_total_derived']}")

    # Check 4: total within the mRALE range.
    total = record["mrale_total_annotated"]
    if total is not None and not (0 <= total <= 24):
        add_exception(record, "total_out_of_range", f"mRALE Score={total} not in [0,24]")

    # Check 5: PCR label vocabulary.
    if record["covid_positive"] not in {"Yes", "No"}:
        add_exception(record, "covid_label_invalid",
                      f"covid_positive={record['covid_positive']!r}")

exceptions_frame = pd.DataFrame(exceptions, columns=["csv_line", "filename", "check", "detail"])
print("Label exceptions by check:")
if len(exceptions_frame):
    for check, count in exceptions_frame["check"].value_counts().items():
        print(f"  {check}: {count}")
    print()
    print(exceptions_frame.head(20).to_string(index=False))
else:
    print("  none")

### A note on the empty extent/density strings

Roughly 590 rows carry an empty `extent_*`/`density_*` text value with a numerical value of
0. That is the encoding for "no opacity", not missing data — an extent of 0 has no
percentage band to name. `EXTENT_TO_NUMERIC[""] = 0` and `DENSITY_TO_NUMERIC[""] = 0` encode
that reading explicitly, so `qualitative_numeric_mismatch` stays quiet for genuine zeros and
fires only when a named band disagrees with its number. If the count of `""` rows differs
materially from the count of `severity_band == "none"` rows, that assumption is wrong and the
cell below will say so.

In [ ]:
zero_extent_text = int((frame["extent_right_text"] == "").sum())
zero_extent_numeric = int((frame["extent_right_numerical"] == 0).sum())
print(f"extent_right_text == '' : {zero_extent_text}")
print(f"extent_right_numerical == 0 : {zero_extent_numeric}")
if zero_extent_text != zero_extent_numeric:
    print()
    print("MISMATCH: the empty-string-means-zero assumption does not hold cleanly. "
          "Inspect the rows below before trusting any mRALE metric.")
    disagreeing = frame[(frame["extent_right_text"] == "") != (frame["extent_right_numerical"] == 0)]
    print(disagreeing[["filename", "extent_right_text", "extent_right_numerical",
                       "mrale_total_annotated"]].head(20).to_string(index=False))
else:
    print("Consistent: empty extent text corresponds exactly to numerical 0.")

## 4. Image QC and hashing

One pass over every PNG. This is the slowest cell in Stage A besides NB 04 (a few minutes
for ~2,600 images) and it is checkpointed: re-running skips images already hashed.

`dhash64` is a 64-bit difference hash computed with PIL only, so no `imagehash` dependency is
introduced. It is used for near-duplicate detection inside MIDRC and, in NB 03, to prove that
external cohorts contain no internal images.

In [ ]:
HASH_CHECKPOINT = NB01_DIR / "image_hashes.jsonl"


def dhash64(image, hash_size=8):
    # Difference hash: compare each pixel with its right-hand neighbour on a 9x8 grey grid.
    small = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.asarray(small, dtype=np.int16)
    bits = pixels[:, 1:] > pixels[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return f"{value:016x}"


def hamming64(left, right):
    return bin(int(left, 16) ^ int(right, 16)).count("1")


def load_checkpoint(path):
    rows = {}
    if not Path(path).is_file():
        return rows
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                rows[row["filename"]] = row
            except Exception:
                continue
    return rows


def append_jsonl(path, row):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


hash_rows = load_checkpoint(HASH_CHECKPOINT)
print(f"Resuming with {len(hash_rows):,} previously hashed images.")

for index, record in enumerate(records, start=1):
    if record["filename"] in hash_rows:
        continue
    row = {"filename": record["filename"], "image_path": record["image_path"]}
    if not record["path_resolved"]:
        row.update({"status": "MISSING", "error": "path did not resolve"})
    else:
        try:
            with Image.open(record["image_path"]) as handle:
                handle.load()
                grey = handle.convert("L")
                statistics = ImageStat.Stat(grey)
                row.update({
                    "status": "OK",
                    "width": handle.width,
                    "height": handle.height,
                    "mode": handle.mode,
                    "format": handle.format,
                    "n_frames": getattr(handle, "n_frames", 1),
                    "mean_intensity": round(statistics.mean[0], 3),
                    "stddev_intensity": round(statistics.stddev[0], 3),
                    "min_intensity": statistics.extrema[0][0],
                    "max_intensity": statistics.extrema[0][1],
                    "dhash64": dhash64(handle),
                })
            row["file_bytes"] = Path(record["image_path"]).stat().st_size
            row["sha256"] = sha256_file(record["image_path"])
        except Exception as exc:
            row.update({"status": "UNREADABLE", "error": f"{type(exc).__name__}: {exc}"})
    append_jsonl(HASH_CHECKPOINT, row)
    hash_rows[record["filename"]] = row
    if index % 200 == 0 or index == len(records):
        print(f"  [{index}/{len(records)}] hashed")

print()
print("Status:", dict(Counter(row.get("status") for row in hash_rows.values())))

In [ ]:
hash_frame = pd.DataFrame(list(hash_rows.values()))
frame = frame.merge(
    hash_frame.drop(columns=["image_path"], errors="ignore"),
    on="filename", how="left", suffixes=("", "_img"),
)
frame["image_status"] = frame["status"].fillna("MISSING")

frame["aspect_ratio"] = frame.apply(
    lambda row: (max(row["width"], row["height"]) / min(row["width"], row["height"]))
    if pd.notna(row.get("width")) and pd.notna(row.get("height")) and min(row["width"], row["height"]) > 0
    else np.nan,
    axis=1,
)
frame["min_side"] = frame[["width", "height"]].min(axis=1)

qc_flags = []
for _, row in frame.iterrows():
    flags = []
    if row["image_status"] != "OK":
        flags.append(f"image_{row['image_status'].lower()}")
    else:
        if pd.notna(row["min_side"]) and row["min_side"] < MIN_IMAGE_SIDE:
            flags.append("small_image")
        if pd.notna(row["aspect_ratio"]) and row["aspect_ratio"] > MAX_ASPECT_RATIO:
            flags.append("extreme_aspect_ratio")
        if pd.notna(row.get("stddev_intensity")) and row["stddev_intensity"] < BLANK_IMAGE_STDDEV:
            flags.append("near_blank")
    if row["quality_issue"]:
        flags.append(f"annotated_quality::{row['quality_issue']}")
    qc_flags.append(";".join(flags))
frame["qc_flags"] = qc_flags

print("QC flag frequency:")
flag_counter = Counter()
for value in frame["qc_flags"]:
    for flag in filter(None, value.split(";")):
        flag_counter[flag] += 1
for flag, count in flag_counter.most_common():
    print(f"  {flag}: {count}")
if not flag_counter:
    print("  none")

print()
if (frame["image_status"] == "OK").any():
    readable = frame[frame["image_status"] == "OK"]
    print("Image dimensions (readable images):")
    print(readable[["width", "height", "min_side", "aspect_ratio"]].describe().to_string())

## 5. Duplicates and near-duplicates

Three distinct notions, kept separate because they have different consequences:

- **Identical filename** — a fatal data error; the manifest key would not be unique.
- **Identical file bytes (sha256)** — the same PNG stored twice. Must not straddle a fold.
- **Near-duplicate pixels (dhash64 within Hamming 3)** — most often two frontal views of the
  same study, which is exactly what the NB 02 grouping key exists to keep together.

In [ ]:
duplicate_filenames = [name for name, count in Counter(frame["filename"]).items() if count > 1]
print("Duplicate filenames:", len(duplicate_filenames), duplicate_filenames[:10])

by_sha = defaultdict(list)
for _, row in frame[frame["image_status"] == "OK"].iterrows():
    by_sha[row["sha256"]].append(row["filename"])
identical_groups = {key: names for key, names in by_sha.items() if len(names) > 1}
print(f"Byte-identical image groups: {len(identical_groups)}")
for key, names in list(identical_groups.items())[:5]:
    print(f"  {key[:16]}...: {names}")

readable = frame[frame["image_status"] == "OK"].reset_index(drop=True)

# Banded LSH prefilter. Splitting the 64-bit hash into 4 bands of 16 bits gives COMPLETE
# recall for Hamming distance <= 3: by the pigeonhole principle, 3 differing bits cannot
# touch all 4 bands, so any true near-duplicate pair shares at least one band exactly.
# (The earlier single 32-bit prefix silently missed pairs whose differences straddled the
# high half of the hash.)
N_BANDS = 4
BAND_HEX = 16 // N_BANDS  # 4 hex characters = 16 bits per band
assert NEAR_DUPLICATE_HAMMING < N_BANDS, (
    f"Banded LSH with {N_BANDS} bands guarantees complete recall only for Hamming "
    f"< {N_BANDS}. Raise N_BANDS or lower NEAR_DUPLICATE_HAMMING."
)

buckets = defaultdict(list)
for index, row in readable.iterrows():
    digest = row["dhash64"]
    for band in range(N_BANDS):
        buckets[(band, digest[band * BAND_HEX:(band + 1) * BAND_HEX])].append(index)

candidate_pairs = set()
for indices in buckets.values():
    if len(indices) > 1:
        for position, left_index in enumerate(indices):
            for right_index in indices[position + 1:]:
                candidate_pairs.add((left_index, right_index))
print(f"Banded LSH candidate pairs to score: {len(candidate_pairs):,}")

near_duplicate_pairs = []
for left_index, right_index in sorted(candidate_pairs):
    left, right = readable.loc[left_index], readable.loc[right_index]
    distance = hamming64(left["dhash64"], right["dhash64"])
    if distance <= NEAR_DUPLICATE_HAMMING:
        if True:
            near_duplicate_pairs.append({
                "filename_a": left["filename"], "filename_b": right["filename"],
                "hamming": distance,
                "sha256_identical": left["sha256"] == right["sha256"],
                "same_study_dir": left["study_dir"] == right["study_dir"],
                "same_covid_label": left["covid_positive"] == right["covid_positive"],
                "covid_a": left["covid_positive"], "covid_b": right["covid_positive"],
                "same_dimensions": (left["width"], left["height"]) == (right["width"], right["height"]),
                "mrale_a": left["mrale_total_annotated"],
                "mrale_b": right["mrale_total_annotated"],
                "same_mrale": left["mrale_total_annotated"] == right["mrale_total_annotated"],
            })

near_duplicate_frame = pd.DataFrame(near_duplicate_pairs)
print()
print(f"Near-duplicate pairs (Hamming <= {NEAR_DUPLICATE_HAMMING}): {len(near_duplicate_frame)}")
if len(near_duplicate_frame):
    cross_study = near_duplicate_frame[~near_duplicate_frame["same_study_dir"]]
    print(f"  within the same study directory: "
          f"{int(near_duplicate_frame['same_study_dir'].sum())}")
    print(f"  ACROSS different study directories: {len(cross_study)}")
    if len(cross_study):
        print()
        print("Cross-study near-duplicates matter: the study directory is NB 02's grouping "
              "key, so a near-duplicate pair in two different directories can still be "
              "split across folds. Review these and consider merging their groups.")
        print(cross_study.head(20).to_string(index=False))
    near_duplicate_frame.to_csv(NB01_DIR / "near_duplicate_pairs.csv", index=False)

### 5b. Is the perceptual hash trustworthy on this cohort?

Chest radiographs are a visually homogeneous class: every frontal CXR shares the same gross
structure, so a 64-bit difference hash computed on an 8x8 grey grid may collide between
genuinely different patients. Before any near-duplicate pair is allowed to influence the
fold-grouping key, the hash's discriminative power is measured directly.

The decisive test is the **null distribution**: sample random image pairs, which are almost
all genuinely different patients, and ask how often they land within the threshold. If random
pairs frequently fall at Hamming <= 3, the detector is producing collisions and its pairs must
not be trusted to merge groups.

The cross-study pairs are then characterised on independent evidence -- byte-identical file
hash, identical dimensions, label agreement -- so that "true re-export" and "hash collision"
can be told apart rather than assumed.


In [ ]:
# --- Null distribution: how often do random (mostly unrelated) pairs fall within threshold?
NULL_SAMPLE_PAIRS = 200_000
digests = readable["dhash64"].tolist()
rng_null = random.Random(SEED)
null_distances = []
if len(digests) > 1:
    for _ in range(NULL_SAMPLE_PAIRS):
        left_index = rng_null.randrange(len(digests))
        right_index = rng_null.randrange(len(digests))
        if left_index == right_index:
            continue
        null_distances.append(hamming64(digests[left_index], digests[right_index]))

hash_diagnostics = {}
if null_distances:
    null_array = np.asarray(null_distances)
    at_or_below = int((null_array <= NEAR_DUPLICATE_HAMMING).sum())
    false_positive_rate = at_or_below / len(null_array)
    n_pairs_total = len(readable) * (len(readable) - 1) / 2
    hash_diagnostics = {
        "null_pairs_sampled": len(null_array),
        "null_mean_hamming": round(float(null_array.mean()), 2),
        "null_sd_hamming": round(float(null_array.std()), 2),
        "null_min_hamming": int(null_array.min()),
        "null_p01_hamming": int(np.percentile(null_array, 1)),
        "null_rate_at_or_below_threshold": round(false_positive_rate, 8),
        "expected_false_pairs_in_cohort": round(false_positive_rate * n_pairs_total, 1),
        "observed_near_duplicate_pairs": len(near_duplicate_frame),
    }
    print("dhash64 null distribution over random image pairs")
    print(f"  pairs sampled            : {len(null_array):,}")
    print(f"  mean Hamming distance    : {null_array.mean():.2f} (sd {null_array.std():.2f})")
    print(f"  minimum observed          : {int(null_array.min())}")
    print(f"  1st percentile            : {int(np.percentile(null_array, 1))}")
    print(f"  P(Hamming <= {NEAR_DUPLICATE_HAMMING})           : {false_positive_rate:.2e}")
    print()
    print(f"  All-pairs in cohort       : {n_pairs_total:,.0f}")
    print(f"  Expected FALSE pairs      : {false_positive_rate * n_pairs_total:.1f}")
    print(f"  Observed pairs            : {len(near_duplicate_frame)}")
    print()
    expected_false = false_positive_rate * n_pairs_total
    if expected_false > 0.5 * max(len(near_duplicate_frame), 1):
        print("  VERDICT: the expected number of chance pairs is a large share of the "
              "observed pairs.")
        print("  The perceptual hash is NOT discriminative enough on this cohort. Set")
        print('  NEAR_DUPLICATE_MERGE_POLICY = "sha256_only" and re-run from Section 6.')
    else:
        print("  VERDICT: chance pairs cannot explain the observed count, so the detected "
              "pairs are")
        print("  predominantly real visual duplicates. Merging them is justified.")

# --- Characterise the cross-study pairs on independent evidence.
if len(near_duplicate_frame):
    cross = near_duplicate_frame[~near_duplicate_frame["same_study_dir"]]
    print()
    print(f"Cross-study near-duplicate pairs: {len(cross)}")
    if len(cross):
        print("  Hamming distribution     :",
              dict(sorted(Counter(cross["hamming"]).items())))
        print(f"  byte-identical (sha256)  : {int(cross['sha256_identical'].sum())} "
              f"({cross['sha256_identical'].mean():.1%})")
        print(f"  identical dimensions     : {int(cross['same_dimensions'].sum())} "
              f"({cross['same_dimensions'].mean():.1%})")
        print(f"  same PCR label           : {int(cross['same_covid_label'].sum())} "
              f"({cross['same_covid_label'].mean():.1%})")
        print(f"  same mRALE total         : {int(cross['same_mrale'].sum())} "
              f"({cross['same_mrale'].mean():.1%})")
        conflicting_pairs = cross[~cross["same_covid_label"]]
        print()
        print(f"  Pairs with DISAGREEING PCR labels: {len(conflicting_pairs)}")
        if len(conflicting_pairs):
            print("  These are the pairs that would manufacture mixed-label groups. Under the")
            print(f'  "{NEAR_DUPLICATE_MERGE_POLICY}" policy they are handled as described in '
                  "Section 6.")
            print(conflicting_pairs[[
                "filename_a", "filename_b", "hamming", "sha256_identical",
                "covid_a", "covid_b", "mrale_a", "mrale_b",
            ]].head(20).to_string(index=False))
            conflicting_pairs.to_csv(
                NB01_DIR / "near_duplicate_label_conflicts.csv", index=False)
        hash_diagnostics.update({
            "cross_study_pairs": len(cross),
            "cross_study_sha256_identical": int(cross["sha256_identical"].sum()),
            "cross_study_same_dimensions": int(cross["same_dimensions"].sum()),
            "cross_study_same_covid_label": int(cross["same_covid_label"].sum()),
            "cross_study_label_disagreeing": len(conflicting_pairs),
        })

print()
print("Interpretation guide:")
print("  high sha256-identical rate  -> genuine re-exports of the same file; merge them")
print("  Hamming 0 but sha256 differs -> same pixels, different encoding; still a duplicate")
print("  labels disagree             -> either an annotation error or a hash collision;")
print("                                 either way, do not force them into one stratum")


## 6. The grouping key for NB 02

There is no patient identifier in `covid_midrc_dataset.csv`. The strongest available
grouping key is the study directory (the DICOM-derived UID in the image path). Grouping at
study level prevents two frontal views of the same acquisition from landing on opposite
sides of a fold boundary.

This is a real limitation and the manuscript must state it: study-level grouping does not
protect against the same *patient* contributing two different studies. If MIDRC case-level
identifiers can be exported, add them as a `patient_id` column and NB 02 will prefer it
automatically.

In [ ]:
frame["group_key"] = frame["study_dir"]
if "patient_id" in frame.columns and frame["patient_id"].notna().all():
    frame["group_key"] = "patient::" + frame["patient_id"].astype(str)
    print("Using patient_id as the grouping key.")
else:
    print("Using study directory as the grouping key (no patient_id column present).")

# STUDY-LEVEL conflicts are checked BEFORE any merging, because a single study directory
# holding both a PCR-positive and a PCR-negative image is a genuine data defect, whereas a
# conflict that appears only after near-duplicate merging is an artefact of the merge.
study_level_labels = frame.groupby("group_key")["covid_positive"].nunique()
study_level_conflicts = study_level_labels[study_level_labels > 1]
print(f"Study-level PCR label conflicts (pre-merge): {len(study_level_conflicts)}")
if len(study_level_conflicts):
    print("  These are real data defects and must be resolved at source:")
    print(frame[frame["group_key"].isin(study_level_conflicts.index)][
        ["filename", "study_uid", "covid_positive", "mrale_total_annotated"]
    ].head(20).to_string(index=False))

parent = {key: key for key in frame["group_key"].unique()}


def find(key):
    while parent[key] != key:
        parent[key] = parent[parent[key]]
        key = parent[key]
    return key


def union(left, right):
    left_root, right_root = find(left), find(right)
    if left_root != right_root:
        parent[max(left_root, right_root)] = min(left_root, right_root)


# Labels currently attached to each group root, so a merge can be tested before it is made.
labels_by_root = defaultdict(set)
for _, row in frame.iterrows():
    labels_by_root[find(row["group_key"])].add(row["covid_positive"])

merge_log = []
merged_pairs = 0
skipped_conflicts = 0

if NEAR_DUPLICATE_MERGE_POLICY == "none":
    print()
    print('Merge policy "none": group_id is exactly the grouping key. Cross-study duplicate '
          'pairs may straddle folds; NB 02 reports them.')
elif len(near_duplicate_frame):
    group_by_filename = dict(zip(frame["filename"], frame["group_key"]))
    cross_pairs = near_duplicate_frame[~near_duplicate_frame["same_study_dir"]]

    if NEAR_DUPLICATE_MERGE_POLICY == "sha256_only":
        cross_pairs = cross_pairs[cross_pairs["sha256_identical"]]
        print()
        print(f'Merge policy "sha256_only": considering {len(cross_pairs)} byte-identical '
              "cross-study pairs.")

    for _, pair in cross_pairs.iterrows():
        left = group_by_filename.get(pair["filename_a"])
        right = group_by_filename.get(pair["filename_b"])
        if not left or not right:
            continue
        left_root, right_root = find(left), find(right)
        if left_root == right_root:
            continue

        combined = labels_by_root[left_root] | labels_by_root[right_root]
        would_conflict = len(combined) > 1

        if would_conflict and NEAR_DUPLICATE_MERGE_POLICY == "conservative":
            skipped_conflicts += 1
            merge_log.append({
                "filename_a": pair["filename_a"], "filename_b": pair["filename_b"],
                "hamming": pair["hamming"], "sha256_identical": pair["sha256_identical"],
                "action": "skipped_would_create_label_conflict",
                "labels": "|".join(sorted(combined)),
            })
            continue

        union(left, right)
        new_root = find(left)
        labels_by_root[new_root] = combined
        for stale_root in {left_root, right_root} - {new_root}:
            labels_by_root.pop(stale_root, None)
        merged_pairs += 1
        merge_log.append({
            "filename_a": pair["filename_a"], "filename_b": pair["filename_b"],
            "hamming": pair["hamming"], "sha256_identical": pair["sha256_identical"],
            "action": "merged_with_label_conflict" if would_conflict else "merged",
            "labels": "|".join(sorted(combined)),
        })

frame["group_id"] = frame["group_key"].map(find)

merge_log_frame = pd.DataFrame(
    merge_log,
    columns=["filename_a", "filename_b", "hamming", "sha256_identical", "action", "labels"],
)
merge_log_frame.to_csv(NB01_DIR / "near_duplicate_merge_log.csv", index=False)

print()
print(f'Merge policy            : "{NEAR_DUPLICATE_MERGE_POLICY}"')
print(f"Group pairs merged      : {merged_pairs}")
print(f"Merges skipped (conflict): {skipped_conflicts}")
if skipped_conflicts:
    print()
    print(f"  {skipped_conflicts} cross-study near-duplicate pair(s) were NOT merged because "
          "the merge would")
    print("  have produced a group containing both a PCR-positive and a PCR-negative image.")
    print("  Such a pair is either an annotation error or a perceptual-hash collision; in")
    print("  neither case is forcing it into one stratum the right call. The pairs are listed")
    print("  in near_duplicate_merge_log.csv and near_duplicate_label_conflicts.csv.")
    print("  NB 02 additionally verifies that these images do not straddle folds and reports")
    print("  any that do.")

group_sizes = frame["group_id"].value_counts()
print()
print(f"Images: {len(frame):,}")
print(f"Unique study directories: {frame['study_dir'].nunique():,}")
print(f"Final grouping units (group_id): {frame['group_id'].nunique():,}")
print("Images per group:", dict(sorted(Counter(group_sizes.values).items())))

label_conflicts = frame.groupby("group_id")["covid_positive"].nunique()
conflicting = label_conflicts[label_conflicts > 1]
merge_induced_conflicts = len(conflicting) - len(study_level_conflicts)
print()
print(f"Groups with mixed PCR labels (post-merge): {len(conflicting)}")
print(f"  of which study-level (real data defect) : {len(study_level_conflicts)}")
print(f"  of which merge-induced                  : {max(0, merge_induced_conflicts)}")
if len(conflicting):
    conflict_detail = frame[frame["group_id"].isin(conflicting.index)][
        ["filename", "group_id", "study_uid", "covid_positive", "mrale_total_annotated"]
    ]
    conflict_detail.to_csv(NB01_DIR / "group_label_conflicts.csv", index=False)
    print()
    print("  NB 02 resolves mixed-label groups by majority vote for STRATIFICATION ONLY.")
    print("  Per-image PCR labels are untouched and are what every metric is computed from,")
    print("  so this affects fold balance, not ground truth.")
    print(conflict_detail.head(20).to_string(index=False))


## 7. Exclusions

The primary cohort keeps every readable image with valid labels. `QUALITY_FLAGS_TO_EXCLUDE`
is empty by default: the annotated quality issues (missing side, bad exposure, other QA) are
recorded and available as a sensitivity analysis, because excluding 59 of 2,581 images from
a cohort this size is a decision that should be justified by its effect, not assumed.

In [ ]:
exception_lines = set(exceptions_frame["csv_line"]) if len(exceptions_frame) else set()
FATAL_CHECKS = {
    "component_missing", "component_out_of_range", "total_missing",
    "total_not_derivable", "total_mismatch", "total_out_of_range", "covid_label_invalid",
}
fatal_lines = set(
    exceptions_frame[exceptions_frame["check"].isin(FATAL_CHECKS)]["csv_line"]
) if len(exceptions_frame) else set()

exclusion_reasons = []
for _, row in frame.iterrows():
    reasons = []
    if row["image_status"] != "OK":
        reasons.append(f"image_{row['image_status'].lower()}")
    if row["csv_line"] in fatal_lines:
        reasons.append("label_integrity_failure")
    if row["quality_issue"] in QUALITY_FLAGS_TO_EXCLUDE:
        reasons.append(f"quality_issue::{row['quality_issue']}")
    exclusion_reasons.append(";".join(reasons))
frame["exclusion_reasons"] = exclusion_reasons
frame["in_primary_cohort"] = frame["exclusion_reasons"] == ""

excluded = frame[~frame["in_primary_cohort"]]
print(f"Primary cohort: {int(frame['in_primary_cohort'].sum()):,} of {len(frame):,}")
print(f"Excluded: {len(excluded):,}")
if len(excluded):
    print(dict(Counter(excluded["exclusion_reasons"])))

excluded[[
    "csv_line", "filename", "image_path", "exclusion_reasons", "qc_flags",
    "covid_positive", "mrale_total_annotated",
]].to_csv(NB01_DIR / "excluded_cases_log.csv", index=False)

primary = frame[frame["in_primary_cohort"]]
print()
print("Primary cohort composition")
print("  PCR:", dict(primary["covid_positive"].value_counts()))
print("  severity band:", dict(primary["severity_band"].value_counts()))
print(f"  groups: {primary['group_id'].nunique():,}")
print(f"  mRALE mean: {primary['mrale_total_annotated'].mean():.3f}, "
      f"sd: {primary['mrale_total_annotated'].std():.3f}")
print()
print("Severity-band counts drive the E8e sampling arm and the severity-stratified MAE "
      "column of Table 2. The sparse bands are the ones to watch.")

## 8. Write the manifest and the quality report

In [ ]:
MANIFEST_COLUMNS = [
    "filename", "image_path", "source_path", "path_resolved", "image_status",
    "study_uid", "study_dir", "group_id", "group_key",
    "covid_positive", "sex", "race", "ethnicity", "quality_issue",
    "extent_right_text", "extent_right_numerical",
    "density_right_text", "density_right_numerical",
    "extent_left_text", "extent_left_numerical",
    "density_left_text", "density_left_numerical",
    "mrale_right", "mrale_left", "mrale_total_derived", "mrale_total_annotated",
    "severity_band", "legacy_fold",
    "width", "height", "min_side", "aspect_ratio", "mode", "format",
    "mean_intensity", "stddev_intensity", "min_intensity", "max_intensity",
    "file_bytes", "sha256", "dhash64",
    "qc_flags", "exclusion_reasons", "in_primary_cohort", "csv_line",
]
manifest = frame.reindex(columns=[c for c in MANIFEST_COLUMNS if c in frame.columns])
manifest_path = NB01_DIR / "midrc_manifest.csv"
manifest.to_csv(manifest_path, index=False)

hash_frame.reindex(columns=["filename", "image_path", "status", "width", "height",
                            "dhash64", "sha256", "file_bytes"]).to_csv(
    NB01_DIR / "image_hashes.csv", index=False)
exceptions_frame.to_csv(NB01_DIR / "label_consistency_exceptions.csv", index=False)

summary = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "01_data_inventory_qc_and_labels.ipynb",
    "source_csv": str(MIDRC_CSV),
    "source_csv_sha256": actual_hash,
    "seed": SEED,
    "thresholds": {
        "min_image_side": MIN_IMAGE_SIDE,
        "max_aspect_ratio": MAX_ASPECT_RATIO,
        "near_duplicate_hamming": NEAR_DUPLICATE_HAMMING,
        "near_duplicate_merge_policy": NEAR_DUPLICATE_MERGE_POLICY,
        "blank_image_stddev": BLANK_IMAGE_STDDEV,
        "quality_flags_excluded": sorted(QUALITY_FLAGS_TO_EXCLUDE),
    },
    "counts": {
        "csv_rows": len(frame),
        "path_resolved": int(frame["path_resolved"].sum()),
        "image_ok": int((frame["image_status"] == "OK").sum()),
        "primary_cohort": int(frame["in_primary_cohort"].sum()),
        "excluded": int((~frame["in_primary_cohort"]).sum()),
        "unique_study_dirs": int(frame["study_dir"].nunique()),
        "grouping_units": int(frame["group_id"].nunique()),
        "duplicate_filenames": len(duplicate_filenames),
        "byte_identical_groups": len(identical_groups),
        "near_duplicate_pairs": len(near_duplicate_frame),
        "near_duplicate_pairs_cross_study": int(
            (~near_duplicate_frame["same_study_dir"]).sum()) if len(near_duplicate_frame) else 0,
        "groups_merged_by_near_duplicate": merged_pairs,
        "merges_skipped_label_conflict": skipped_conflicts,
        "study_level_label_conflicts": len(study_level_conflicts),
        "groups_mixed_label_post_merge": len(conflicting),
        "label_exceptions": len(exceptions_frame),
        "label_exceptions_fatal": len(fatal_lines),
    },
    "distributions": {
        "covid_positive": {str(k): int(v) for k, v in frame["covid_positive"].value_counts().items()},
        "severity_band": {str(k): int(v) for k, v in frame["severity_band"].value_counts().items()},
        "sex": {str(k): int(v) for k, v in frame["sex"].value_counts().items()},
        "race": {str(k): int(v) for k, v in frame["race"].value_counts().items()},
        "ethnicity": {str(k): int(v) for k, v in frame["ethnicity"].value_counts().items()},
        "quality_issue": {str(k or "<none>"): int(v)
                          for k, v in frame["quality_issue"].value_counts().items()},
        "images_per_group": {str(k): int(v)
                             for k, v in Counter(group_sizes.values).items()},
    },
    "mrale": {
        "mean": float(primary["mrale_total_annotated"].mean()),
        "sd": float(primary["mrale_total_annotated"].std()),
        "min": int(primary["mrale_total_annotated"].min()),
        "max": int(primary["mrale_total_annotated"].max()),
        "pcr_positive_prevalence": float((primary["covid_positive"] == "Yes").mean()),
    },
    "hash_diagnostics": hash_diagnostics,
    "limitations": [
        "No patient identifier is available; grouping is at study level. Two studies from "
        "the same patient can therefore still fall in different folds.",
        "Empty extent/density text is interpreted as the value 0 (no opacity). Section 3 "
        "verifies that this reading is internally consistent.",
    ],
}
with (NB01_DIR / "inventory_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

report_lines = [
    "# MIDRC data-quality report (Stage A / NB 01)",
    "",
    f"Generated: {summary['written_utc']}",
    f"Source: `{MIDRC_CSV.name}` (sha256 `{actual_hash[:16]}...`)",
    "",
    "## Counts",
    "",
    "| item | value |",
    "| --- | --- |",
]
for key, value in summary["counts"].items():
    report_lines.append(f"| {key.replace('_', ' ')} | {value:,} |")
report_lines += ["", "## PCR and severity distribution", "",
                 "| stratum | n |", "| --- | --- |"]
for key, value in summary["distributions"]["covid_positive"].items():
    report_lines.append(f"| PCR {key} | {value:,} |")
for key, value in summary["distributions"]["severity_band"].items():
    report_lines.append(f"| severity {key} | {value:,} |")
report_lines += [
    "",
    f"PCR-positive prevalence: **{summary['mrale']['pcr_positive_prevalence']:.4f}**.",
    "",
    "## Label exceptions", "",
]
if len(exceptions_frame):
    report_lines += ["| check | n |", "| --- | --- |"]
    for check, count in exceptions_frame["check"].value_counts().items():
        report_lines.append(f"| {check} | {count:,} |")
else:
    report_lines.append("None.")
report_lines += ["", "## Limitations", ""]
report_lines += [f"- {item}" for item in summary["limitations"]]
(NB01_DIR / "data_quality_report.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print("Wrote:")
for name in ["midrc_manifest.csv", "image_hashes.csv", "label_consistency_exceptions.csv",
             "excluded_cases_log.csv", "inventory_summary.json", "data_quality_report.md"]:
    print("  ", NB01_DIR / name)

## 9. Gate

In [ ]:
# Set True only after reviewing label_consistency_exceptions.csv and accepting that the
# affected rows are excluded from the primary cohort.
ACKNOWLEDGE_LABEL_EXCLUSIONS = False

failures = []
warnings = []

unresolved = int((~frame["path_resolved"]).sum())
if unresolved:
    failures.append(f"{unresolved} image paths did not resolve after rewriting.")

unreadable = int((frame["image_status"] == "UNREADABLE").sum())
if unreadable:
    failures.append(f"{unreadable} images could not be opened.")

if duplicate_filenames:
    failures.append(f"{len(duplicate_filenames)} duplicate filenames; the manifest key is "
                    "not unique.")

if len(fatal_lines):
    failures.append(
        f"{len(fatal_lines)} rows failed a fatal label check "
        f"({sorted(FATAL_CHECKS)}). They are excluded from the primary cohort and listed in "
        "label_consistency_exceptions.csv; confirm the exclusion is acceptable, then set "
        "ACKNOWLEDGE_LABEL_EXCLUSIONS = True at the top of this cell and re-run."
    )

# Study-level conflicts are a genuine data defect and block the gate. Conflicts created by
# near-duplicate merging are an artefact of this notebook's own grouping step and are handled
# by NB 02's majority-vote stratification, so they warn rather than block.
if len(study_level_conflicts):
    failures.append(
        f"{len(study_level_conflicts)} study directories contain both a PCR-positive and a "
        "PCR-negative image. That is a source-data defect, not a grouping artefact: one "
        "acquisition cannot have two PCR results. Resolve it in covid_midrc_dataset.csv "
        "before building folds. See group_label_conflicts.csv."
    )

residual_mixed = len(conflicting) - len(study_level_conflicts)
if residual_mixed > 0:
    warnings.append(
        f"{residual_mixed} grouping unit(s) hold mixed PCR labels after near-duplicate "
        f'merging under policy "{NEAR_DUPLICATE_MERGE_POLICY}". NB 02 assigns each group a '
        "single stratum label by majority vote for fold balancing only; per-image labels and "
        "all metrics are unaffected. Review near_duplicate_label_conflicts.csv to decide "
        'whether to switch to NEAR_DUPLICATE_MERGE_POLICY = "sha256_only".'
    )

if skipped_conflicts:
    warnings.append(
        f"{skipped_conflicts} cross-study near-duplicate pair(s) were left unmerged because "
        "the merge would have created a mixed-label group. NB 02 checks that these images do "
        "not straddle folds."
    )

if hash_diagnostics.get("expected_false_pairs_in_cohort") is not None:
    expected = hash_diagnostics["expected_false_pairs_in_cohort"]
    observed = max(hash_diagnostics.get("observed_near_duplicate_pairs", 0), 1)
    if expected > 0.5 * observed:
        warnings.append(
            f"Perceptual-hash null test: {expected:.1f} chance pairs expected versus "
            f"{observed} observed. dhash64 is weakly discriminative on this cohort; prefer "
            'NEAR_DUPLICATE_MERGE_POLICY = "sha256_only".'
        )

minimum_cohort = 2000
if int(frame["in_primary_cohort"].sum()) < minimum_cohort:
    failures.append(f"Primary cohort below {minimum_cohort} images; check the exclusions.")

if len(near_duplicate_frame) and int((~near_duplicate_frame["same_study_dir"]).sum()):
    cross_count = int((~near_duplicate_frame["same_study_dir"]).sum())
    warnings.append(
        f"{cross_count} near-duplicate pairs span different study directories; "
        f"{merged_pairs} group pair(s) were merged and {skipped_conflicts} skipped. "
        "Review near_duplicate_pairs.csv and near_duplicate_merge_log.csv."
    )

if flag_counter:
    warnings.append(f"QC flags present: {dict(flag_counter)}")

if ACKNOWLEDGE_LABEL_EXCLUSIONS:
    failures = [item for item in failures if "fatal label check" not in item]
    warnings.append("Label-integrity exclusions acknowledged by the operator.")


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

with (NB01_DIR / "gate_nb01.json").open("w", encoding="utf-8") as handle:
    json.dump({"passed": not failures, "failures": failures, "warnings": warnings},
              handle, indent=2)

assert not failures, f"NB 01 gate failed with {len(failures)} blocking issue(s)."
print()
print("NB 01 gate: PASSED")
print(f"Primary cohort: {int(frame['in_primary_cohort'].sum()):,} images in "
      f"{primary['group_id'].nunique():,} grouping units.")
print(f"Merge policy: {NEAR_DUPLICATE_MERGE_POLICY} "
      f"({merged_pairs} merged, {skipped_conflicts} skipped)")

## Notes carried forward

- `midrc_manifest.csv` is the single input to NB 02. NB 02 must not re-read
  `covid_midrc_dataset.csv`.
- `group_id` is the fold-grouping unit: study directory, with cross-study near-duplicate
  pairs merged. NB 02 groups on this column and nothing else.
- `dhash64` and `sha256` are reused by NB 03 for the external-cohort de-duplication audit
  and by the conditional X4 overlap check.
- `in_primary_cohort` defines the denominator for every internal metric in the paper. If it
  ever changes, Table 1 and every downstream count change with it, so change it here and
  nowhere else.